In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing base dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    
    if not os.path.exists("models/benchmark/ConLID/repo"):
        print("Setting up ConLID dependencies...")
        os.makedirs("models/benchmark/ConLID", exist_ok=True)
        os.system('git clone https://github.com/epfl-nlp/language-identification.git models/benchmark/ConLID/repo')
        os.system('pip install -q -r models/benchmark/ConLID/repo/requirements.txt')
        
    print("Setup complete!")


In [2]:
import torch

print(torch.__version__)
print(torch.__file__)
print(torch.cuda.is_available())

2.8.0+cpu
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\torch\__init__.py
False


In [3]:
# NOTE: If running this notebook manually in the IDE, make sure to select the `conlid-venv` kernel!
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'

In [4]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "eng_Latn",
    "sin": "sin_Sinh",

    # Do NOT put "san" here.
    # Sanskrit is handled separately using its script.

    "tam": "tam_Taml",
    "hin": "hin_Deva",
    "ben": "ben_Beng",
    "arb": "arb_Arab",
    "fra": "fra_Latn",
    "deu": "deu_Latn",
}

def detect_script(text):
    """
    Detect the script used by the text.
    """

    text = str(text)

    has_sinhala = any(
        "\u0D80" <= ch <= "\u0DFF"
        for ch in text
    )

    has_devanagari = any(
        "\u0900" <= ch <= "\u097F"
        for ch in text
    )

    if has_sinhala and not has_devanagari:
        return "Sinh"

    if has_devanagari and not has_sinhala:
        return "Deva"

    if has_sinhala and has_devanagari:
        return "Mixed"

    return "Unknown"


def map_true_label(row):

    label = str(row.get("label", "")).strip()
    source = str(row.get("source", "")).strip()

    if label == "san":

        # Sanskrit added by our project in Sinhala script
        if source in [
            "DCS",
            "SansinNT",
            "SiDiaC-v2"
        ]:
            return "san_Sinh"

        # Sanskrit coming from the original benchmark
        return "san_Deva"

    return TARGET_LANGUAGES.get(label)

def load_dataset(file_path):

    print(
        f"\nLoading {os.path.basename(file_path)}..."
    )

    records = []

    with open(file_path, encoding="utf-8") as f:

        for line in f:

            row = json.loads(line)

            raw_label = row.get("label")

            # Keep normal target languages
            # AND keep all Sanskrit examples.
            if (
                raw_label in TARGET_LANGUAGES
                or raw_label == "san"
            ):
                records.append(row)


    df = pd.DataFrame(records)


    if not df.empty:

        # Correct script-aware mapping
        df["flores_label"] = df.apply(
            map_true_label,
            axis=1
        )

        # Only remove Sanskrit rows whose script
        # genuinely could not be determined.
        df = df[
            df["flores_label"].notna()
        ].copy()


        print(
            f"Loaded {len(df)} rows across "
            f"{df['flores_label'].nunique()} "
            f"language-script classes"
        )


        print("\nTrue-label counts:")

        print(
            df["flores_label"]
            .value_counts()
            .sort_index()
        )


    else:

        print(
            "No matching target languages "
            "found in this dataset."
        )


    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [5]:
import os
import sys

REPO_DIR = "models/benchmark/ConLID/repo"

if not os.path.exists(REPO_DIR):
    raise RuntimeError(
        f"{REPO_DIR} not found. "
        "Please run 'make setup-conlid' in the pipeline root first."
    )

print("Python being used:")
print(sys.executable)

# Make sure only the ConLID source code is added.
# Do not add another virtual environment.
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

# Check PyTorch from the CURRENT notebook environment
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())

from model import ConLID
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

print("Downloading ConLID checkpoints...")
checkpoint_dir = os.path.join(REPO_DIR, "checkpoints", "conlid")
snapshot_download(repo_id="epfl-nlp/ConLID", local_dir=checkpoint_dir)

print("Loading ConLID model...")
conlid_model = ConLID.from_pretrained(dir=checkpoint_dir)
model_name = "ConLID"
target_labels = [
    "arb_Arab",
    "ben_Beng",
    "deu_Latn",
    "eng_Latn",
    "fra_Latn",
    "hin_Deva",
    "san_Deva",
    "san_Sinh",
    "sin_Sinh",
    "tam_Taml",
]

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty:
        continue
    
    print(f"Running ConLID inference on {len(df)} rows...")
    predictions = []
    texts = df["text"].tolist()
    
    for text in tqdm(texts, desc=f"Processing {dataset_name}"):
        try:
            res = conlid_model.predict(text)
            if isinstance(res, (list, tuple)) and len(res) > 0:
                first = res[0]
                if isinstance(first, (list, tuple)) and len(first) > 0:
                    pred_lang = str(first[0])
                else:
                    pred_lang = str(first)
            else:
                pred_lang = str(res)
        except Exception as e:
            pred_lang = "unknown"
        predictions.append(pred_lang)
        
    df["predicted_label"] = predictions
    df["true_label"] = df["flores_label"]
    
    evaluate_and_save(df, model_name, dataset_name, target_labels)


Python being used:
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Scripts\python.exe
PyTorch version: 2.8.0+cpu
PyTorch location: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\torch\__init__.py
CUDA available: False


Returning existing local_dir `D:\Projects\ML Projects\LangID - DSE project\data_pipeline\models\benchmark\ConLID\repo\checkpoints\conlid` as remote repo cannot be accessed in `snapshot_download` ([WinError 10054] An existing connection was forcibly closed by the remote host).


Loading ConLID model...

Loading commonlid.jsonl...
Loaded 74947 rows across 10 language-script classes

True-label counts:
flores_label
arb_Arab    26152
ben_Beng     1886
deu_Latn     7553
eng_Latn    27461
fra_Latn     3233
hin_Deva     3666
san_Deva      895
san_Sinh     1327
sin_Sinh     2693
tam_Taml       81
Name: count, dtype: int64
Running ConLID inference on 74947 rows...


Processing commonlid:   0%|          | 0/74947 [00:00<?, ?it/s]


ZERO-SHOT BENCHMARK RESULTS (ConLID on commonlid)
Accuracy:  72.10%
Macro F1:  80.47%

Per-language breakdown:



...

              precision    recall  f1-score   support

    arb_Arab     0.9957    0.2297    0.3733      2024
    ben_Beng     1.0000    0.9990    0.9995      1012
    deu_Latn     1.0000    0.9941    0.9970      1012
    eng_Latn     1.0000    0.9921    0.9960      1012
    fra_Latn     1.0000    1.0000    1.0000      1012
    hin_Deva     0.9970    0.9970    0.9970      1012
    san_Deva     1.0000    0.9970    0.9985      1012
    san_Sinh     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6654    0.9770    0.7916      2693
    tam_Taml     0.9990    1.0000    0.9995      1012

   micro avg     0.8843    0.7738    0.8254     13128
   macro avg     0.8657    0.8186    0.8153     13128
weighted avg     0.8293    0.7738    0.7586     13128


Saved predictions to datasets/benchmark_results\conlid_flores_plus.csv


Loading wili-2018.jsonl...
Loaded 11020 rows across 9 language-script classes

True-label counts:
flores_label
ben_Beng    1000
deu_Latn    1000
eng_Latn    1000
fra_L

Processing wili-2018:   0%|          | 0/11020 [00:00<?, ?it/s]


ZERO-SHOT BENCHMARK RESULTS (ConLID on wili-2018)
Accuracy:  85.08%
Macro F1:  76.08%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.0000    0.0000    0.0000         0
    ben_Beng     1.0000    0.8930    0.9435      1000
    deu_Latn     0.9989    0.9440    0.9707      1000
    eng_Latn     0.9146    0.9750    0.9439      1000
    fra_Latn     0.9859    0.9780    0.9819      1000
    hin_Deva     1.0000    0.9800    0.9899      1000
    san_Deva     1.0000    0.9850    0.9924      1000
    san_Sinh     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6654    0.9770    0.7916      2693
    tam_Taml     0.9990    0.9900    0.9945      1000

   micro avg     0.8675    0.8508    0.8591     11020
   macro avg     0.7564    0.7722    0.7608     11020
weighted avg     0.7886    0.8508    0.8120     11020



d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
 


Saved predictions to datasets/benchmark_results\conlid_wili-2018.csv



In [6]:
import torch

print("PyTorch version:", torch.__version__)
print("PyTorch location:", torch.__file__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.8.0+cpu
PyTorch location: d:\Projects\ML Projects\LangID - DSE project\data_pipeline\.venv\Lib\site-packages\torch\__init__.py
CUDA available: False


In [7]:
print("\nTrue class counts:")
print(
    df["flores_label"]
    .value_counts()
    .sort_index()
)


True class counts:
flores_label
ben_Beng    1000
deu_Latn    1000
eng_Latn    1000
fra_Latn    1000
hin_Deva    1000
san_Deva    1000
san_Sinh    1327
sin_Sinh    2693
tam_Taml    1000
Name: count, dtype: int64
